# MNIST tour of `jarl.experiments`

<div style="padding: 0.75rem 1rem; border-left: 4px solid #2563eb; margin: 1rem 0;">
<strong>Goal.</strong> Build a multi-branch MNIST experiment tree — baseline, hyperparameter forks, branch extensions, and checkouts — while exercising the public <code>jarl.experiments</code> API.
</div>


## Prerequisites

- Python 3.12: `uv sync --extra examples`
- Run from `examples/notebooks/`
- MNIST downloads once under `./_data` (~60 MB)
- Optional: TensorBoard via `tracking.track_tensorboard=True`


## What you will learn

- Grow a **multi-branch experiment tree** from one baseline root
- Fork with different configs (LR, width, batch size) from a pinned checkpoint
- Extend completed branches and navigate with **`checkout`**
- Compare branch heads, lineage metrics, and DAG plots
- Recovery, deferred preparation, artifacts, JIT callbacks, and disk reload


In [ ]:
from __future__ import annotations

import shutil
import sys
from pathlib import Path

import jax
import matplotlib.pyplot as plt
import numpy as np

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name != "notebooks":
    NOTEBOOK_DIR = (NOTEBOOK_DIR / "examples" / "notebooks").resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

EXP_DIR = NOTEBOOK_DIR / "_outputs" / "mnist_experiments_tour"
DATA_ROOT = NOTEBOOK_DIR / "_data"

if EXP_DIR.exists():
    shutil.rmtree(EXP_DIR)
EXP_DIR.mkdir(parents=True, exist_ok=True)

print(f"Experiment directory: {EXP_DIR}")

In [ ]:
from jarl.experiments import (
    CHECKPOINT_ALIAS_BEST,
    CHECKPOINT_ALIAS_FINAL,
    CheckpointRef,
    ExperimentGraph,
    JsonlMetricReader,
    TrackingConfig,
    plot_dag,
)
from jarl.experiments.run_config import CheckpointConfig, RunConfig
from notebook_utils import (
    evaluate_mnist,
    load_mnist_arrays,
    load_warmstart_params,
    print_branch_summary,
    resolve_fork_step,
    train_from_config,
)

TRAIN_BATCH, TEST_BATCH = load_mnist_arrays(train_size=2048, test_size=512, data_root=str(DATA_ROOT))
print(f"Train: {TRAIN_BATCH.images.shape}, Test: {TEST_BATCH.images.shape}")

## 1. Config and baseline root

We mirror the TFM MNIST DAG demo: a short baseline on `main`, then forks with sparse config overrides. `max_to_keep=8` keeps the fork checkpoint on disk after root training finishes.


In [ ]:
class MNISTExperimentConfig(RunConfig):
    """MNIST demo config for the experiments tour."""

    DEFAULT_PROJECT_NAME = "jarl-demos"
    DEFAULT_EXP_NAME = "mnist-experiments-tour"
    learning_rate: float = 0.02
    hidden_dim: int = 64
    batch_size: int = 128
    train_steps: int = 12
    seed: int = 0


base_config = MNISTExperimentConfig(
    checkpoint=CheckpointConfig(
        max_to_keep=8,
        save_interval_steps=4,
    ),
    tracking=TrackingConfig(track_tensorboard=False, track_wandb=False),
)

graph = ExperimentGraph(EXP_DIR, base_config=base_config)
root = graph.create_root(
    config=base_config,
    branch="main",
    label="baseline",
    description="Root MNIST baseline (TFM-style)",
    metadata={"dataset": "MNIST", "demo": "experiments-tour"},
)

with root:
    train_from_config(root, base_config, TRAIN_BATCH, TEST_BATCH)

on_disk = sorted(int(path.name) for path in root.checkpoint_dir.iterdir() if path.name.isdigit())
print(root)
print("Registry checkpoints:", [r.checkpoint_step for r in root.list_checkpoints()])
print("On-disk checkpoints:", on_disk)
print("Final alias:", root.resolve_checkpoint_alias(CHECKPOINT_ALIAS_FINAL))

## 2. Hyperparameter forks from a pinned checkpoint

All sweeps fork from an **early baseline checkpoint** (step 4 when still on disk). We resolve the step dynamically because Orbax retention can drop registry entries that are no longer restorable when `max_to_keep` is low.

<div style="padding: 0.75rem 1rem; border-left: 4px solid #f59e0b; background: #fffbeb; margin: 1rem 0;">
<strong>Note.</strong> Changing <code>hidden_dim</code> requires a fresh parameter init — the fork still records lineage/config, but training cannot warm-start incompatible weights.
</div>

```
main (baseline)
├── lr_low      lr=0.01
├── lr_high     lr=0.05
├── wide        hidden_dim=128 (fresh init)
└── small_batch batch_size=64
```


In [ ]:
PREFERRED_FORK_STEP = 4
FORK_STEP = resolve_fork_step(root, preferred=PREFERRED_FORK_STEP)
print(f"Using fork step {FORK_STEP} (preferred {PREFERRED_FORK_STEP})")
root.pin_checkpoint(FORK_STEP)
fork_ref = CheckpointRef(node_id=root.id, checkpoint_step=FORK_STEP)

# warmstart=False when architecture changes (e.g. hidden_dim)
FORK_SPECS: list[tuple[str, str, MNISTExperimentConfig, str, bool]] = [
    (
        "lr_low",
        "lr_0.01",
        base_config.model_copy(update={"learning_rate": 0.01, "train_steps": 8}),
        "TFM reference: lower LR fork",
        True,
    ),
    (
        "lr_high",
        "lr_0.05",
        base_config.model_copy(update={"learning_rate": 0.05, "train_steps": 8}),
        "Aggressive LR fork",
        True,
    ),
    (
        "wide",
        "h128",
        base_config.model_copy(update={"hidden_dim": 128, "train_steps": 8}),
        "Wider MLP (fresh init)",
        False,
    ),
    (
        "small_batch",
        "bs64",
        base_config.model_copy(update={"batch_size": 64, "train_steps": 8}),
        "Smaller minibatches",
        True,
    ),
]

fork_nodes: dict[str, object] = {}
for branch, label, cfg, description, warmstart in FORK_SPECS:
    node = graph.fork(
        branch,
        from_node=root,
        config=cfg,
        label=label,
        description=description,
        from_checkpoint=fork_ref,
    )
    warmstart_params = load_warmstart_params(node) if warmstart else None
    with node:
        train_from_config(
            node,
            cfg,
            TRAIN_BATCH,
            TEST_BATCH,
            params=warmstart_params,
        )
    fork_nodes[branch] = node
    print(f"{branch}: {node.id} -> acc={node.latest_metrics()['test_accuracy']:.3f}")

## 3. Extend branches (immutable completed nodes)

Completed nodes stay read-only. To continue on the same branch, **`extend`** appends a child that inherits the parent checkpoint.


In [ ]:
main_fast_cfg = base_config.model_copy(update={"learning_rate": 0.03, "train_steps": 6})
main_fast = graph.extend("main", config=main_fast_cfg, label="fast_child", description="Higher LR on main")
with main_fast:
    train_from_config(
        main_fast,
        graph.resolve_config(main_fast),
        TRAIN_BATCH,
        TEST_BATCH,
        params=load_warmstart_params(main_fast, from_parent=True),
    )

lr_low_fine_cfg = graph.resolve_config(fork_nodes["lr_low"]).model_copy(
    update={"learning_rate": 0.003, "train_steps": 6},
)
lr_low_fine = graph.extend(
    "lr_low",
    config=lr_low_fine_cfg,
    label="fine_tune",
    description="Fine-tune the low-LR branch",
)
with lr_low_fine:
    train_from_config(
        lr_low_fine,
        graph.resolve_config(lr_low_fine),
        TRAIN_BATCH,
        TEST_BATCH,
        params=load_warmstart_params(lr_low_fine, from_parent=True),
    )

print("Main head:", graph.head("main").id)
print("lr_low head:", graph.head("lr_low").id)

## 4. Checkout tour

`checkout` moves the **current node** pointer (like `git checkout`) without changing branch heads. Useful to inspect configs and metrics anywhere in the tree.


In [ ]:
checkout_targets = [
    ("baseline root", root),
    ("lr_low head", graph.head("lr_low")),
    ("lr_high head", graph.head("lr_high")),
    ("wide head", graph.head("wide")),
    ("main tip", graph.head("main")),
]

for title, node in checkout_targets:
    graph.checkout(node)
    cfg = graph.resolve_config(graph.current_node)
    diff = graph.get_config_diff(root, graph.current_node)
    metrics = graph.current_node.latest_metrics()
    print(f"\n=== checkout: {title} ===")
    print("current:", graph.current_node.id)
    print("branch:", graph.current_node.branch)
    print("lr:", cfg.learning_rate, "hidden:", cfg.hidden_dim, "batch:", cfg.batch_size)
    print("diff vs root:", diff.to_overrides() if diff else {})
    print("test_accuracy:", metrics.get("test_accuracy"))

print("\nLineage to main tip:", [n.id for n in graph.get_lineage(graph.head("main"))])
print("Lineage metrics:", graph.get_metrics_along_lineage(graph.head("main")))

## 5. Branch summary and DAG visualization

Compare every branch head side-by-side, then plot the full tree colored by `test_accuracy`.


In [ ]:
print_branch_summary(graph)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
plot_dag(graph, metric_key="test_accuracy", ax=axes[0])
axes[0].set_title("Full experiment tree")

plot_dag(graph, metric_key="test_accuracy", ax=axes[1])
axes[1].set_title(f"Current node: {graph.current_node.id}")
plt.tight_layout()
plt.show()

## 6. Deferred preparation (`prepare=True`)


In [ ]:
prepared_cfg = graph.resolve_config(graph.head("lr_low")).model_copy(update={"train_steps": 4})
prepared = graph.extend("lr_low", config=prepared_cfg, label="prepared_arm", prepare=True)

print("Prepared status:", prepared.status)
print("Has metrics file:", prepared.metrics_jsonl_path.exists())

with prepared:
    train_from_config(prepared, prepared_cfg, TRAIN_BATCH, TEST_BATCH)

print("After explicit start:", prepared.status)
print(JsonlMetricReader(prepared.metrics_jsonl_path).latest())

## 7. Failure, resume, and execution attempts

Keep the `try/except` **outside** the `with` block so the workspace records a failed attempt instead of marking the node completed.


In [ ]:
recovery = graph.fork(
    "recovery",
    from_node=root,
    label="fail_and_resume",
    config=base_config.model_copy(update={"train_steps": 8}),
    from_checkpoint=fork_ref,
)

try:
    with recovery:
        train_from_config(
            recovery,
            graph.resolve_config(recovery),
            TRAIN_BATCH,
            TEST_BATCH,
            params=load_warmstart_params(recovery),
            fail_at_step=4,
        )
except RuntimeError as exc:
    print("Expected failure:", exc)

print("Failed status:", recovery.status)

with recovery:
    resumed_state = recovery.load_resume_checkpoint()
    train_from_config(
        recovery,
        graph.resolve_config(recovery),
        TRAIN_BATCH,
        TEST_BATCH,
        params=resumed_state["params"],
        start_step=4,
        register_model=False,
    )

print("Attempts:", [(a.attempt_id, a.status.value, a.resume_of_attempt_id) for a in recovery.list_execution_attempts()])

## 8. Checkpoints, artifacts, and disk reload


In [ ]:
best_step = max(root.list_checkpoints(), key=lambda r: r.metrics.get("test_accuracy", 0.0)).checkpoint_step
root.promote_checkpoint_best(
    best_step,
    metric_name="test_accuracy",
    metric_value=root.latest_metrics()["test_accuracy"],
    reason="notebook demo",
)
print("Best checkpoint alias:", root.resolve_checkpoint_alias(CHECKPOINT_ALIAS_BEST))
print("Root artifacts:", root.list_artifacts())

graph.save()
reloaded = ExperimentGraph.from_directory(EXP_DIR, config_cls=MNISTExperimentConfig)
print("Reloaded nodes:", len(reloaded.all_nodes))
print("Branches after reload:", list(reloaded.get_branches()))
print_branch_summary(reloaded)

## 9. Model archives and JIT metric callbacks


In [ ]:
from typing import Any

import jax.numpy as jnp
import optax

from jarl.experiments import MODEL_ALIAS_BEST, make_scalar_metric_log_callback
from notebook_utils import MNISTMLP, init_mnist_params, sample_minibatch

loaded = root.load_model_archive("policy", base_config.train_steps)
print("Reloaded accuracy:", evaluate_mnist(loaded["params"], TEST_BATCH, hidden_dim=base_config.hidden_dim))
root.promote_model_best(
    "policy",
    base_config.train_steps,
    metric_name="test_accuracy",
    metric_value=root.latest_metrics()["test_accuracy"],
    reason="notebook demo",
)
print("Model best alias:", root.resolve_model_alias(MODEL_ALIAS_BEST))

jit_demo = graph.fork("jit_demo", from_node=root, label="jit_metrics", config=base_config)
model = MNISTMLP(hidden_dim=base_config.hidden_dim)
params = init_mnist_params(jax.random.key(7), hidden_dim=base_config.hidden_dim)
optimizer = optax.adam(0.01)
opt_state = optimizer.init(params)
rng = np.random.default_rng(7)

with jit_demo:
    host_log = make_scalar_metric_log_callback(jit_demo.log_scalar)

    @jax.jit
    def _train_step_jit(
        step: int,
        step_params: dict[str, Any],
        step_opt_state: optax.OptState,
        x_batch: jax.Array,
        y_batch: jax.Array,
    ) -> tuple[dict[str, Any], optax.OptState]:
        def loss_fn(current_params: dict[str, Any]) -> jax.Array:
            logits = model.apply(current_params, x_batch)
            return optax.softmax_cross_entropy_with_integer_labels(logits, y_batch).mean()

        loss, grads = jax.value_and_grad(loss_fn)(step_params)
        updates, next_opt_state = optimizer.update(grads, step_opt_state)
        next_params = optax.apply_updates(step_params, updates)
        jax.debug.callback(host_log, step, "train_loss", loss)
        return next_params, next_opt_state

    for step in range(1, 4):
        x_np, y_np = sample_minibatch(TRAIN_BATCH, batch_size=64, rng=rng)
        params, opt_state = _train_step_jit(step, params, opt_state, jnp.asarray(x_np), jnp.asarray(y_np))
    print("JIT metrics:", JsonlMetricReader(jit_demo.metrics_jsonl_path).latest())

## 10. TensorBoard helpers and final tree


In [ ]:
from jarl.experiments import tensorboard_compare, tensorboard_lineage

tb_config = base_config.model_copy(update={"tracking": TrackingConfig(track_tensorboard=True, track_wandb=False)})
tb_node = graph.fork("tb_demo", from_node=root, label="tensorboard", config=tb_config)
with tb_node:
    train_from_config(tb_node, tb_config, TRAIN_BATCH, TEST_BATCH, train_steps=2, register_model=False)

print("Lineage spec:", tensorboard_lineage(graph, graph.head("lr_low")))
print("Compare spec:", tensorboard_compare(graph, ["main", "lr_low", "lr_high", "wide"]))
print("NetworkX:", graph.as_networkx().number_of_nodes(), "nodes,", graph.as_networkx().number_of_edges(), "edges")

fig = plot_dag(graph, metric_key="test_accuracy", figsize=(14, 10))
plt.show()

## Takeaways

- One **baseline root** can spawn many **config forks** from the same pinned checkpoint.
- **`extend`** continues a branch; **`fork`** opens a new one — completed nodes stay immutable.
- **`checkout`** inspects any node without moving branch heads.
- **`print_branch_summary`** + **`plot_dag`** give a quick read of the full sweep.
- Recovery, artifacts, JIT callbacks, and `from_directory` round out the workspace IO surface.

## References

- TFM reference: `learning/mnist/mnist_dag_training.ipynb`
- `src/jarl/experiments/graph.py` — tree coordinator API
- `src/jarl/experiments/node.py` — workspace lifecycle and IO
